# 🏕️ Caravan Detection with Detectron2

Train Mask R-CNN to detect caravans in satellite/aerial imagery.

## Before you start:
1. **Enable GPU**: Runtime → Change runtime type → T4 GPU
2. **Upload your dataset to Google Drive** in this structure:
```
MyDrive/
└── caravan_dataset/
    ├── train/
    │   ├── images/     (your .tif or .png files)
    │   └── labels/1/   (instance mask .png files)
    └── val/
        ├── images/
        └── labels/1/
```

## Step 1: Check GPU

In [ ]:
# Check if GPU is available
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU found! Go to Runtime → Change runtime type → T4 GPU")

## Step 2: Install Detectron2

In [ ]:
# Install Detectron2
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

# Verify installation
import detectron2
print(f"Detectron2 version: {detectron2.__version__}")

## Step 3: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Check if your dataset folder exists
import os

# ⚠️ CHANGE THIS PATH to match your Google Drive folder
DATASET_PATH = "/content/drive/MyDrive/caravan_dataset"

if os.path.exists(DATASET_PATH):
    print(f"✅ Dataset found at: {DATASET_PATH}")
    print(f"   Train images: {len(os.listdir(os.path.join(DATASET_PATH, 'train/images')))} files")
    print(f"   Val images: {len(os.listdir(os.path.join(DATASET_PATH, 'val/images')))} files")
else:
    print(f"❌ Dataset NOT found at: {DATASET_PATH}")
    print("Please upload your dataset to Google Drive and update DATASET_PATH above")

## Step 4: Setup Dataset & Model

In [ ]:
import os
import glob
import numpy as np
import cv2
import random
import matplotlib.pyplot as plt

from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.utils.visualizer import Visualizer, ColorMode
from detectron2.structures import BoxMode
from detectron2.evaluation import COCOEvaluator

# For proper mask encoding
import pycocotools.mask as mask_util


def get_caravan_dicts(dataset_dir, subset):
    """
    Load caravan dataset in Detectron2 format.
    Uses bitmask format for segmentation (more reliable than polygons).
    """
    subset_dir = os.path.join(dataset_dir, subset)
    image_dir = os.path.join(subset_dir, "images")
    mask_dir = os.path.join(subset_dir, "labels", "1")

    dataset_dicts = []

    # Find all images
    image_files = []
    for ext in ['*.tif', '*.tiff', '*.png', '*.jpg', '*.jpeg']:
        image_files.extend(glob.glob(os.path.join(image_dir, ext)))
        image_files.extend(glob.glob(os.path.join(image_dir, ext.upper())))

    print(f"Found {len(image_files)} images in {subset} set")

    total_instances = 0

    for idx, image_path in enumerate(image_files):
        record = {}

        # Read image to get dimensions
        image = cv2.imread(image_path)
        if image is None:
            print(f"Warning: Could not read {image_path}, skipping...")
            continue

        height, width = image.shape[:2]

        record["file_name"] = image_path
        record["image_id"] = idx
        record["height"] = height
        record["width"] = width

        # Find corresponding mask file
        image_name = os.path.basename(image_path)
        base_name = os.path.splitext(image_name)[0]
        mask_path = None
        for ext in ['.png', '.PNG', '.tif', '.tiff', '.TIF']:
            potential_mask = os.path.join(mask_dir, base_name + ext)
            if os.path.exists(potential_mask):
                mask_path = potential_mask
                break

        annotations = []

        if mask_path and os.path.exists(mask_path):
            # Read instance mask - handle both single-channel and multi-channel
            mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)

            if mask is None:
                print(f"Warning: Could not read mask {mask_path}")
                record["annotations"] = []
                dataset_dicts.append(record)
                continue

            # Convert to single channel if needed
            if len(mask.shape) == 3:
                # If RGB/RGBA, convert to grayscale or use first channel
                if mask.shape[2] == 4:
                    mask = mask[:, :, 0]  # Use first channel
                else:
                    mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)

            # Get unique instance IDs (excluding 0 which is background)
            instance_ids = np.unique(mask)
            instance_ids = instance_ids[instance_ids > 0]

            for instance_id in instance_ids:
                # Create binary mask for this instance
                binary_mask = (mask == instance_id).astype(np.uint8)

                # Check mask has actual content
                if binary_mask.sum() < 10:  # Skip tiny masks
                    continue

                # Get bounding box from mask
                rows = np.any(binary_mask, axis=1)
                cols = np.any(binary_mask, axis=0)
                if not np.any(rows) or not np.any(cols):
                    continue

                rmin, rmax = np.where(rows)[0][[0, -1]]
                cmin, cmax = np.where(cols)[0][[0, -1]]

                # Bbox in XYXY format
                bbox = [int(cmin), int(rmin), int(cmax), int(rmax)]

                # Skip degenerate boxes
                if bbox[2] <= bbox[0] or bbox[3] <= bbox[1]:
                    continue

                # Convert mask to COCO RLE format (most reliable for Detectron2)
                # Mask must be Fortran-contiguous
                binary_mask_fortran = np.asfortranarray(binary_mask)
                rle = mask_util.encode(binary_mask_fortran)
                rle["counts"] = rle["counts"].decode("utf-8")  # JSON serializable

                annotation = {
                    "bbox": bbox,
                    "bbox_mode": BoxMode.XYXY_ABS,
                    "segmentation": rle,  # RLE format instead of polygon
                    "category_id": 0,
                    "iscrowd": 0,
                }
                annotations.append(annotation)
                total_instances += 1

        record["annotations"] = annotations
        dataset_dicts.append(record)

    print(f"Loaded {len(dataset_dicts)} images with {total_instances} total instances from {subset} set")
    return dataset_dicts


def register_caravan_dataset(dataset_dir):
    """Register the caravan dataset with Detectron2."""
    for subset in ["train", "val"]:
        dataset_name = f"caravan_{subset}"

        # Remove if already registered
        if dataset_name in DatasetCatalog.list():
            DatasetCatalog.remove(dataset_name)
            MetadataCatalog.remove(dataset_name)

        # Register dataset
        DatasetCatalog.register(
            dataset_name,
            lambda d=dataset_dir, s=subset: get_caravan_dicts(d, s)
        )
        MetadataCatalog.get(dataset_name).set(thing_classes=["caravan"])

    print("✅ Dataset registered successfully!")


# Register the dataset
register_caravan_dataset(DATASET_PATH)

# ============================================
# DEBUG: Inspect dataset and masks
# ============================================
# Run this cell to verify your masks are loading correctly!

print("🔍 Inspecting dataset...")
dataset_dicts = get_caravan_dicts(DATASET_PATH, "train")

# Show stats
total_annotations = sum(len(d["annotations"]) for d in dataset_dicts)
images_with_annotations = sum(1 for d in dataset_dicts if len(d["annotations"]) > 0)

print(f"\n📊 Dataset Statistics:")
print(f"   Total images: {len(dataset_dicts)}")
print(f"   Images with annotations: {images_with_annotations}")
print(f"   Total caravan instances: {total_annotations}")
print(f"   Avg instances per image: {total_annotations / max(len(dataset_dicts), 1):.1f}")

# Show a single annotation example
if dataset_dicts and dataset_dicts[0]["annotations"]:
    print(f"\n📝 Example annotation structure:")
    example = dataset_dicts[0]["annotations"][0]
    print(f"   bbox: {example['bbox']}")
    print(f"   bbox_mode: {example['bbox_mode']}")
    print(f"   category_id: {example['category_id']}")
    print(f"   segmentation type: {type(example['segmentation'])}")
    if isinstance(example['segmentation'], dict):
        print(f"   segmentation keys: {example['segmentation'].keys()}")
        print(f"   segmentation size: {example['segmentation'].get('size', 'N/A')}")
    print("\n✅ Segmentation is in RLE format (correct for Detectron2)")
else:
    print("\n⚠️ WARNING: No annotations found! Check your mask files.")

In [ ]:
# Visualize training samples with ground truth masks
caravan_metadata = MetadataCatalog.get("caravan_train")

# Show 3 random samples with their masks
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for i in range(3):
    # Pick a random image with annotations
    samples_with_annos = [d for d in dataset_dicts if len(d["annotations"]) > 0]
    if not samples_with_annos:
        print("No samples with annotations found!")
        break

    d = random.choice(samples_with_annos)
    img = cv2.imread(d["file_name"])
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Top row: original image with GT annotations
    visualizer = Visualizer(img_rgb, metadata=caravan_metadata, scale=0.5)
    vis = visualizer.draw_dataset_dict(d)
    axes[0, i].imshow(vis.get_image())
    axes[0, i].set_title(f"Image + GT: {len(d['annotations'])} caravans")
    axes[0, i].axis('off')

    # Bottom row: show the raw mask
    mask_dir = os.path.join(DATASET_PATH, "train", "labels", "1")
    base_name = os.path.splitext(os.path.basename(d["file_name"]))[0]
    for ext in ['.png', '.PNG', '.tif', '.tiff']:
        mask_path = os.path.join(mask_dir, base_name + ext)
        if os.path.exists(mask_path):
            mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
            if len(mask.shape) == 3:
                mask = mask[:, :, 0]
            axes[1, i].imshow(mask, cmap='tab20')
            axes[1, i].set_title(f"Raw mask: {len(np.unique(mask)) - 1} instances")
            break
    axes[1, i].axis('off')

plt.suptitle("Top: Image with Ground Truth | Bottom: Raw Instance Mask", fontsize=14)
plt.tight_layout()
plt.show()

print("\n💡 Each color in the bottom row = one caravan instance")
print("   If masks look wrong, check your labeling tool output format")

## Step 6: Configure Training

In [ ]:
# Output directory (in Google Drive so it persists)
OUTPUT_DIR = "/content/drive/MyDrive/caravan_model_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)


class CaravanTrainer(DefaultTrainer):
    """Custom trainer with evaluation."""
    @classmethod
    def build_evaluator(cls, cfg, dataset_name):
        return COCOEvaluator(dataset_name, output_dir=OUTPUT_DIR)


# Setup configuration
cfg = get_cfg()

# Use Mask R-CNN with ResNet-50 FPN (faster than R-101, still good)
cfg.merge_from_file(model_zoo.get_config_file(
    "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
))

# Dataset
cfg.DATASETS.TRAIN = ("caravan_train",)
cfg.DATASETS.TEST = ("caravan_val",)

# Use COCO pre-trained weights
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
)

# ============================================
# Training parameters (optimized for fine-tuning)
# ============================================
cfg.DATALOADER.NUM_WORKERS = 2

# Batch size - 2 is good for T4 GPU with 16GB memory
cfg.SOLVER.IMS_PER_BATCH = 2

# Learning rate - LOW because we're fine-tuning from COCO weights
# Higher LR would destroy the pre-trained features
cfg.SOLVER.BASE_LR = 0.0001

# Warmup - gradually increase LR to avoid early training instability
cfg.SOLVER.WARMUP_ITERS = 200
cfg.SOLVER.WARMUP_METHOD = "linear"
cfg.SOLVER.WARMUP_FACTOR = 1.0 / 200

# Total iterations - for 1000 images, 5000 iterations = ~10 epochs
cfg.SOLVER.MAX_ITER = 5000

# LR decay - reduce LR at these steps for final convergence
# Set to empty () initially to diagnose if learning is happening
# Once confirmed working, can set to (3500, 4500) for better final results
cfg.SOLVER.STEPS = (3500, 4500)
cfg.SOLVER.GAMMA = 0.1  # Multiply LR by 0.1 at each step

# Checkpoints - save frequently so you don't lose progress
cfg.SOLVER.CHECKPOINT_PERIOD = 500

# ============================================
# Model parameters
# ============================================
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128  # ROIs per image
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1             # Just "caravan"
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5     # Detection confidence threshold

# Output
cfg.OUTPUT_DIR = OUTPUT_DIR

# Evaluation - check progress every 500 iterations
cfg.TEST.EVAL_PERIOD = 500

print("\n📋 Training Configuration:")
print(f"   Batch size: {cfg.SOLVER.IMS_PER_BATCH}")
print(f"   Learning rate: {cfg.SOLVER.BASE_LR} (low for fine-tuning)")
print(f"   Warmup iterations: {cfg.SOLVER.WARMUP_ITERS}")
print(f"   Max iterations: {cfg.SOLVER.MAX_ITER}")
print(f"   LR decay steps: {cfg.SOLVER.STEPS}")
print(f"   Checkpoint every: {cfg.SOLVER.CHECKPOINT_PERIOD} iterations")
print(f"   Evaluate every: {cfg.TEST.EVAL_PERIOD} iterations")
print(f"   Output directory: {cfg.OUTPUT_DIR}")
print("\n💡 Watch for:")
print("   - Training loss should decrease over time")
print("   - AP50 should increase (target: > 0.3 for decent, > 0.5 for good)")

## Step 7: Train! 🚀

This will take ~1-2 hours for 1000 images with 5000 iterations.

**If disconnected**: Just re-run cells 1-6, then run the "Resume Training" cell below instead.

In [ ]:
# Start training from scratch
trainer = CaravanTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

print("\n✅ Training complete!")
print(f"Model saved to: {OUTPUT_DIR}")

In [ ]:
# Resume training (if disconnected)
# Uncomment and run this cell if you need to resume from a checkpoint

# trainer = CaravanTrainer(cfg)
# trainer.resume_or_load(resume=True)  # This loads the latest checkpoint
# trainer.train()

## Step 8: Test the Model

In [ ]:
# Load the trained model
cfg.MODEL.WEIGHTS = os.path.join(OUTPUT_DIR, "model_final.pth")
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5  # Detection threshold

predictor = DefaultPredictor(cfg)

# Test on validation images
val_dicts = get_caravan_dicts(DATASET_PATH, "val")

# Show results on 6 random validation images
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for ax in axes.flatten():
    d = random.choice(val_dicts)
    img = cv2.imread(d["file_name"])

    # Run detection
    outputs = predictor(img)

    # Visualize
    v = Visualizer(
        img[:, :, ::-1],
        metadata=MetadataCatalog.get("caravan_val"),
        scale=0.5,
        instance_mode=ColorMode.IMAGE_BW
    )
    vis = v.draw_instance_predictions(outputs["instances"].to("cpu"))

    num_detections = len(outputs["instances"])
    ax.imshow(vis.get_image())
    ax.set_title(f"Detected: {num_detections} caravan(s)")
    ax.axis('off')

plt.tight_layout()
plt.show()

## Step 9: Run Detection on New Images

In [ ]:
def detect_caravans(image_path, output_path=None, threshold=0.5):
    """
    Detect caravans in an image and save/display results.
    """
    # Load image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Error: Could not read {image_path}")
        return None

    # Update threshold if needed
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = threshold
    predictor = DefaultPredictor(cfg)

    # Run detection
    outputs = predictor(img)
    instances = outputs["instances"].to("cpu")

    print(f"Found {len(instances)} caravan(s) in {os.path.basename(image_path)}")

    # Visualize
    v = Visualizer(
        img[:, :, ::-1],
        metadata=MetadataCatalog.get("caravan_val"),
        scale=1.0,
        instance_mode=ColorMode.IMAGE_BW
    )
    vis = v.draw_instance_predictions(instances)
    result_img = vis.get_image()

    # Save if output path provided
    if output_path:
        cv2.imwrite(output_path, result_img[:, :, ::-1])
        print(f"Saved to: {output_path}")

    # Display
    plt.figure(figsize=(12, 8))
    plt.imshow(result_img)
    plt.axis('off')
    plt.title(f"Detected {len(instances)} caravan(s)")
    plt.show()

    return instances


# Example: detect on a specific image
# detect_caravans("/content/drive/MyDrive/my_image.tif", threshold=0.5)

In [ ]:
# Batch detection on a folder
def batch_detect(input_folder, output_folder, threshold=0.5):
    """
    Run detection on all images in a folder.
    """
    os.makedirs(output_folder, exist_ok=True)

    # Find images
    image_files = []
    for ext in ['*.tif', '*.png', '*.jpg']:
        image_files.extend(glob.glob(os.path.join(input_folder, ext)))

    print(f"Processing {len(image_files)} images...")

    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = threshold
    predictor = DefaultPredictor(cfg)

    results = []
    for img_path in image_files:
        img = cv2.imread(img_path)
        if img is None:
            continue

        outputs = predictor(img)
        instances = outputs["instances"].to("cpu")

        # Save visualization
        v = Visualizer(img[:, :, ::-1], metadata=MetadataCatalog.get("caravan_val"), scale=1.0)
        vis = v.draw_instance_predictions(instances)

        output_path = os.path.join(output_folder, os.path.basename(img_path).replace('.tif', '_detected.png'))
        cv2.imwrite(output_path, vis.get_image()[:, :, ::-1])

        results.append({
            'image': os.path.basename(img_path),
            'num_caravans': len(instances)
        })
        print(f"  {os.path.basename(img_path)}: {len(instances)} caravans")

    print(f"\n✅ Done! Results saved to: {output_folder}")
    return results


# Example:
# batch_detect(
#     input_folder="/content/drive/MyDrive/new_images",
#     output_folder="/content/drive/MyDrive/detection_results",
#     threshold=0.5
# )

## Step 10: Download Model

Your trained model is automatically saved to Google Drive at:
- `caravan_model_output/model_final.pth` - Final trained model
- `caravan_model_output/model_*.pth` - Checkpoints

To use locally, download `model_final.pth` from your Drive.

In [ ]:
# List saved models
print("Saved models:")
for f in sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.pth"))):
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"  {os.path.basename(f)} ({size_mb:.1f} MB)")

---
## 💡 Tips

**If training is too slow:**
- Reduce `MAX_ITER` (e.g., 3000 instead of 5000)
- Use smaller images if possible

**If detection quality is poor:**
- Increase `MAX_ITER` (e.g., 10000)
- Lower threshold: `cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.3`
- Add more training data

**If you run out of GPU memory:**
- Reduce batch size: `cfg.SOLVER.IMS_PER_BATCH = 1`
- Reduce ROI batch: `cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 64`